In [1]:
from pathlib import Path

import torch

from tfmplayground.benchmarks.forecasting.adapters import NanoTabPFNForecastAdapter
from tfmplayground.benchmarks.forecasting.config import ForecastBenchmarkConfig
from tfmplayground.benchmarks.forecasting.datasets import load_suite
from tfmplayground.benchmarks.forecasting.runner import evaluate_regression

/home/mouad/Desktop/dev_projects/tfm_forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
repo_root = Path.cwd().parent  # if needed, set explicitly to your repo root

model_ckpt = repo_root / "workdir" / "nanoTFM" / "latest_checkpoint.pth"
dist_ckpt = repo_root / "workdir" / "sanity_dynscm_small_buckets.pth"

# quick sanity on checkpoint format
ckpt = torch.load(model_ckpt, map_location="cpu")
print("has architecture:", isinstance(ckpt, dict) and "architecture" in ckpt)

cfg = ForecastBenchmarkConfig.from_dict(
    {
        "mode": "regression",
        "output_dir": str(repo_root / "workdir" / "forecast_results_exchange_sanity"),
        "datasets": {
            "dataset_names": ["exchange_rate"],
            "max_series_per_dataset": 4,
            "allow_download": True,
        },
        "protocol": {
            "horizons": [1, 3, 6],
            "context_rows": 12,
            "test_rows": 6,
            "max_feature_lag": 8,
            "explicit_lags": [0, 1, 2, 4],
            "num_kernels": 1,
        },
    }
)

suite = load_suite(cfg)
adapters = {
    "nanotabpfn_dynscm": NanoTabPFNForecastAdapter(
        name="nanotabpfn_dynscm",
        model_path=str(model_ckpt),
        dist_path=str(dist_ckpt),
        device="cpu",
    )
}

rows = evaluate_regression(cfg, suite=suite, adapters=adapters, device="cpu")
out = Path(cfg.output_dir)
out.mkdir(parents=True, exist_ok=True)
rows.to_csv(out / "exchange_rate_rows.csv", index=False)

ok = rows[rows["status"] == "ok"]
print("ok_rows:", len(ok), "skipped_rows:", int((rows["status"] == "skipped").sum()))
if len(ok):
    print(ok[["rmse", "smape", "mase"]].mean().to_dict())
else:
    print(rows[["status", "skip_reason"]].head(20).to_string(index=False))

has architecture: True
ok_rows: 11 skipped_rows: 0
{'rmse': 0.13392352902334578, 'smape': 0.1564538925055202, 'mase': 22.048727156044748}
